# AGN Optical Variability - DRW Timescale Pipeline

Measure how fast an active galactic nucleus varies in brightness, from raw survey data to a defensible number.

Run the cells in order, top to bottom. Each one prints what it did and why before you move on. Nothing here is tied to a particular object - cell 2 asks you for the target.

---

## Cell 1 - Install and check the libraries

Installs anything missing and prints a version table. Run this once per machine (or once per Colab session).

In [ ]:
# ==============================================================
# CELL 1 - install and verify every library this pipeline needs
# ==============================================================
import importlib
import subprocess
import sys

# (import name, pip name, required?)
REQUIRED = [
    ("numpy",      "numpy",      True),
    ("pandas",     "pandas",     True),
    ("scipy",      "scipy",      True),
    ("matplotlib", "matplotlib", True),
    ("requests",   "requests",   True),
    ("celerite2",  "celerite2",  True),
    ("astroquery", "astroquery", False),   # optional, only used for the SIMBAD lookup in cell 3
]


def _version_of(module_name):
    try:
        mod = importlib.import_module(module_name)
    except Exception:
        return None
    return getattr(mod, "__version__", "unknown")


print("checking what's already installed...\n")

missing = [(m, p, r) for (m, p, r) in REQUIRED if _version_of(m) is None]

if not missing:
    print("nothing missing, everything is already here")
else:
    for module_name, pip_name, required in missing:
        print(f"installing {pip_name} ...")
        proc = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            capture_output=True, text=True,
        )
        if proc.returncode != 0:
            tail = proc.stderr.strip().splitlines()
            print(f"   pip failed for {pip_name}: {tail[-1] if tail else 'no error text'}")
    importlib.invalidate_caches()

print("\n" + "=" * 52)
print(f"{'library':<14}{'version':<18}{'status'}")
print("-" * 52)

all_required_ok = True
for module_name, pip_name, required in REQUIRED:
    version = _version_of(module_name)
    if version is not None:
        status = "OK"
    else:
        status = "MISSING (required)" if required else "missing (optional)"
        if required:
            all_required_ok = False
    print(f"{module_name:<14}{str(version or '-'):<18}{status}")

print("=" * 52)
if all_required_ok:
    print("\nall required libraries are ready - go to cell 2")
    if _version_of("astroquery") is None:
        print("(astroquery missing is fine - cell 3 will just ask you for the")
        print(" object classification instead of looking it up automatically)")
else:
    print("\nsomething required is missing. re-run this cell once; if it still")
    print("fails, read the pip error above and install that package by hand.")

print("\n" + "-" * 52)
print("NOTES FOR YOU")
print("-" * 52)
print("  what this cell does: checks what's already on this machine and pip")
print("     installs only what's missing. safe to re-run any time.")
print()
print("  on Google Colab: if anything was freshly installed, restart the")
print("     runtime (Runtime > Restart session) and run this cell again before")
print("     moving on. Colab sometimes keeps the old version loaded otherwise.")
print()
print("  celerite2 is the one that matters most - it does the Gaussian process")
print("     fitting in cell 8. if it won't install, nothing after cell 7 works.")
print()
print("  astroquery is optional. without it cell 3 simply asks you to type the")
print("     object classification instead of looking it up for you.")
print()
print("  next: cell 2, where you enter your target")

## Cell 2 - Target, cone search parameters, and download

Asks for the object name, the date, the cone search position (RA, Dec and radius in degrees, exactly as the ZTF API wants it), the file format and where to save it. Then prints back everything you gave it and downloads the light curve.

In [ ]:
# ==============================================================
# CELL 2 - target details, cone search parameters, and download
# ==============================================================
import os
from datetime import date
from io import StringIO

import pandas as pd
import requests

ZTF_API_URL = "https://irsa.ipac.caltech.edu/cgi-bin/ZTF/nph_light_curves"


def ask(prompt, default=None, cast=str, validate=None, error=""):
    """keep asking until the answer is usable. Enter accepts the default."""
    while True:
        shown = f"{prompt} [{default}]: " if default is not None else f"{prompt}: "
        answer = input(shown).strip()
        if answer == "" and default is not None:
            answer = str(default)
        if answer == "":
            print("   -> can't leave this blank")
            continue
        try:
            value = cast(answer)
        except Exception:
            print(f"   -> couldn't read that as {cast.__name__}, try again")
            continue
        if validate is not None and not validate(value):
            print(f"   -> {error}")
            continue
        return value


AGN_NAME = ask("AGN / target name")
OBS_DATE = ask("analysis date (YYYY-MM-DD)", default=date.today().isoformat())

print("\ncone search position - the API wants POS = CIRCLE ra dec radius, all in degrees")
RA_DEG = ask("   RA in degrees", cast=float,
             validate=lambda v: 0.0 <= v <= 360.0,
             error="RA has to be between 0 and 360")
DEC_DEG = ask("   Dec in degrees", cast=float,
              validate=lambda v: -90.0 <= v <= 90.0,
              error="Dec has to be between -90 and +90")
RADIUS_DEG = ask("   search radius in degrees", default=0.000417, cast=float,
                 validate=lambda v: 0 < v <= 0.1,
                 error="radius must be > 0 and <= 0.1 deg (0.000417 deg = 1.5 arcsec)")

BAND = ask("band (g / r / i)", default="r",
           validate=lambda v: v.lower() in ("g", "r", "i"),
           error="pick g, r or i").lower()
FILE_FORMAT = ask("file format", default="csv",
                  validate=lambda v: v.lower() == "csv",
                  error="this pipeline reads CSV - type csv").lower()
SAVE_DIR = os.path.expanduser(ask("folder to save the file in", default=os.getcwd()))
os.makedirs(SAVE_DIR, exist_ok=True)

_safe = "".join(c if (c.isalnum() or c in "-_") else "_" for c in AGN_NAME)
CSV_PATH = os.path.join(SAVE_DIR, f"{_safe}_ztf_{BAND}.{FILE_FORMAT}")

PARAMS = {
    "POS": f"CIRCLE {RA_DEG} {DEC_DEG} {RADIUS_DEG}",
    "BANDNAME": BAND,
    "FORMAT": FILE_FORMAT.upper(),
}

print("\n" + "=" * 66)
print("WHAT WE ARE ASKING THE ARCHIVE FOR")
print("=" * 66)
print(f"target             : {AGN_NAME}")
print(f"analysis date      : {OBS_DATE}")
print(f"RA (deg)           : {RA_DEG}")
print(f"Dec (deg)          : {DEC_DEG}")
print(f"search radius (deg): {RADIUS_DEG}    ({RADIUS_DEG * 3600:.2f} arcsec)")
print(f"band               : {BAND}")
print(f"file format        : {FILE_FORMAT}")
print(f"archive            : IRSA ZTF light curve API")
print(f"POS string sent    : {PARAMS['POS']}")
print(f"saving to          : {CSV_PATH}")
print("=" * 66)

print("\ndownloading ...")
try:
    resp = requests.get(ZTF_API_URL, params=PARAMS, timeout=120)
except requests.exceptions.RequestException as exc:
    raise RuntimeError(f"couldn't reach the IRSA/ZTF API: {exc}") from exc

if resp.status_code != 200:
    raise RuntimeError(f"query failed ({resp.status_code}): {resp.text[:300]}")
if not resp.text.strip():
    raise RuntimeError("the archive sent back an empty response - check RA/Dec/band")

with open(CSV_PATH, "w") as f:
    f.write(resp.text)

RAW = pd.read_csv(StringIO(resp.text))
if RAW.empty:
    raise RuntimeError(
        "query went through but came back with zero rows - either the target "
        "isn't in the ZTF footprint, or the coordinates/radius are off"
    )

print(f"got {len(RAW)} rows and {len(RAW.columns)} columns")
print(f"saved to {CSV_PATH}")

print("\n" + "-" * 66)
print("NOTES FOR YOU")
print("-" * 66)
print("  where to get RA and Dec: search the object on NED")
print("     (ned.ipac.caltech.edu) or SIMBAD (simbad.u-strasbg.fr) and copy the")
print("     J2000 position in DEGREES, not hours-minutes-seconds. if your")
print("     source lists 12h29m06.7s you need to convert it first.")
print()
print("  the radius: 0.000417 deg is 1.5 arcsec, the standard choice for a")
print("     point source. too small and slight pointing offsets lose you real")
print("     epochs; too large and you start blending in a neighbouring star.")
print("     only widen it if you get zero rows back.")
print()
print("  the band: r usually has the most epochs and the least host-galaxy")
print("     contamination. g works too. never analyse two bands together -")
print("     they trace different parts of the disc.")
print()
print("  if you got zero rows: check RA/Dec are in degrees and in the right")
print("     order, then check the object is actually in ZTF's sky coverage")
print("     (ZTF is a northern survey - targets below about -30 deg Dec")
print("     simply aren't observed).")
print()
print("  if a lot of rows came back: that's normal, ZTF revisits fields often.")
print("     cell 6 will thin it down to one point per night.")
print()
print("  next: cell 3, the viewing angle screen - do that before any analysis")
print("-" * 66)

## Cell 3 - Viewing angle screen

Before any analysis: is the jet pointed at us? If it is, the optical variability is contaminated by beamed jet emission and the timescale we measure later is not an accretion disc timescale. Returns **LOW**, **NORMAL**, **HIGH** or **UNKNOWN** risk.

In [ ]:
# ==============================================================
# CELL 3 - viewing angle screen: is this target a fair DRW subject?
# ==============================================================
#
# A DRW fit assumes the optical variability comes from the accretion
# disc itself. Two different geometries break that assumption:
#
#   looking straight down the jet  -> relativistic beaming piles
#       non-thermal jet emission on top of the disc, and tau stops
#       being a disc timescale (blazars, BL Lacs, FSRQs)
#   looking through the dusty torus -> the disc is hidden altogether
#       and what varies is scattered or host light (type 2 objects)
#
# The sweet spot is in between: far enough off the jet axis that
# nothing is beamed at us, but not so edge-on that the torus blocks
# the view. We can't measure the jet angle here (that needs VLBI), so
# this uses the standard indirect indicators: source classification,
# radio loudness, radio spectral shape and optical polarization.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Ellipse, Circle, Wedge, Arc, Patch

if "AGN_NAME" not in globals():
    raise RuntimeError("run cell 2 first - the target name isn't set yet")


def simbad_type(name):
    """ask SIMBAD what kind of object this is. returns (type, source)."""
    try:
        from astroquery.simbad import Simbad
    except Exception:
        return None, "astroquery not installed"
    try:
        sim = Simbad()
        try:
            sim.add_votable_fields("otype")
        except Exception:
            pass
        result = sim.query_object(name)
        if result is None or len(result) == 0:
            return None, f"SIMBAD has no match for '{name}'"
        for key in ("OTYPE", "otype", "OTYPE_S", "main_type", "otype_txt"):
            if key in result.colnames:
                value = result[key][0]
                if isinstance(value, bytes):
                    value = value.decode()
                return str(value), "SIMBAD"
        return None, "SIMBAD answered but without a type column"
    except Exception as exc:
        return None, f"SIMBAD lookup failed ({type(exc).__name__})"


# order matters - the obscured patterns are checked before the generic
# "seyfert" ones, otherwise "Seyfert 2" would match as disc-dominated
JET_ALIGNED = ["bllac", "bl lac", "bl_lac", "blazar", "bla", "fsrq",
               "flat-spectrum", "flat spectrum"]
OBSCURED = ["sy2", "sy 2", "seyfert 2", "seyfert2", "seyfert_2", "type 2",
            "type ii", "type-2", "nlrg", "obscured"]
JET_PRESENT = ["radio galaxy", "radiog", "radio-loud", "radio loud",
               "fri", "frii", "compact steep"]
DISC_VISIBLE = ["sy1", "sy 1", "seyfert 1", "seyfert1", "seyfert_1", "type 1",
                "type i", "nls1", "narrow-line seyfert 1", "radio-quiet",
                "radio quiet", "liner", "seyfert"]

print(f"screening {AGN_NAME} ...\n")

auto_type, type_source = simbad_type(AGN_NAME)
if auto_type:
    print(f"SIMBAD classification: {auto_type}")
else:
    print(f"automatic lookup unavailable: {type_source}")

KNOWN_TYPE = input(
    f"object classification [{auto_type or 'press Enter to skip'}]: "
).strip() or (auto_type or "")


def _optional_float(prompt):
    raw = input(prompt).strip()
    if raw == "":
        return None
    try:
        return float(raw)
    except ValueError:
        print("   -> couldn't read that as a number, skipping it")
        return None


print("\nthese are all optional - press Enter to skip any you don't have:")
RADIO_LOUDNESS = _optional_float("   radio loudness R = F_5GHz / F_B: ")
RADIO_ALPHA = _optional_float("   radio spectral index alpha (S ~ nu^alpha): ")
OPT_POL_PCT = _optional_float("   optical polarization in percent: ")

findings = []      # (level, explanation)
IS_OBSCURED = False

if KNOWN_TYPE:
    kt = KNOWN_TYPE.lower()
    if any(k in kt for k in JET_ALIGNED):
        findings.append(("high", f"classified as '{KNOWN_TYPE}' - a jet-aligned class by definition"))
    elif any(k in kt for k in OBSCURED):
        IS_OBSCURED = True
        findings.append(("normal", f"classified as '{KNOWN_TYPE}' - an obscured (edge-on) type. "
                                   f"the torus hides the disc, so the optical light is scattered "
                                   f"or host-dominated rather than the disc itself"))
    elif any(k in kt for k in JET_PRESENT):
        findings.append(("normal", f"classified as '{KNOWN_TYPE}' - has a jet, but not pointed at us"))
    elif any(k in kt for k in DISC_VISIBLE):
        findings.append(("low", f"classified as '{KNOWN_TYPE}' - disc seen directly, jet not beamed at us"))
    else:
        print(f"\n(note: '{KNOWN_TYPE}' isn't specific enough to classify on its own)")

if RADIO_LOUDNESS is not None:
    if RADIO_LOUDNESS > 100:
        findings.append(("high", f"radio loudness R = {RADIO_LOUDNESS:.1f} > 100 - strongly radio-loud"))
    elif RADIO_LOUDNESS > 10:
        findings.append(("normal", f"radio loudness R = {RADIO_LOUDNESS:.1f} - radio-loud, jet contributes"))
    else:
        findings.append(("low", f"radio loudness R = {RADIO_LOUDNESS:.1f} <= 10 - radio-quiet"))

if RADIO_ALPHA is not None:
    if RADIO_ALPHA > -0.5:
        findings.append(("high", f"flat/inverted radio spectrum (alpha = {RADIO_ALPHA:.2f}) - "
                                 f"beamed core, small viewing angle"))
    else:
        findings.append(("low", f"steep radio spectrum (alpha = {RADIO_ALPHA:.2f}) - "
                                f"lobe dominated, larger viewing angle"))

if OPT_POL_PCT is not None:
    if OPT_POL_PCT > 3.0:
        findings.append(("high", f"optical polarization {OPT_POL_PCT:.1f}% > 3% - synchrotron/jet signature"))
    else:
        findings.append(("low", f"optical polarization {OPT_POL_PCT:.1f}% - no strong jet signature"))

levels = [lvl for lvl, _ in findings]
if "high" in levels:
    VIEWING_RISK = "HIGH"
elif "normal" in levels:
    VIEWING_RISK = "NORMAL"
elif "low" in levels:
    VIEWING_RISK = "LOW"
else:
    VIEWING_RISK = "UNKNOWN"

VIEWING_REASONS = [text for _, text in findings]

# the angle we draw: inferred from the verdict, NOT measured.
#
# a classification alone only tells us which ZONE we're probably in - it
# does not pin down a specific angle within that zone. earlier this cell
# drew one fixed number per category (8 / 35 / 45 / 75) and labelled it
# "theta ~= X deg" as if that were derived for this object, which made
# every plain Seyfert 1 (no radio loudness, no spectral index, no optical
# polarization entered) look identical and falsely precise. if you only
# gave a classification, that's genuinely all we know here, so the honest
# thing is to say "theta is somewhere in this range", not to guess a
# single degree. a specific value is only drawn when a quantitative
# indicator actually pins it down.
ZONE_RANGE = {"HIGH": (2.0, 15.0), "OBSCURED": (60.0, 88.0),
              "NORMAL": (15.0, 60.0), "LOW": (15.0, 60.0)}

_zone_key = "OBSCURED" if IS_OBSCURED else VIEWING_RISK
_lo, _hi = ZONE_RANGE.get(_zone_key, (0.0, 90.0))
THETA_ILLUSTRATIVE = (_lo + _hi) / 2.0   # midpoint, used only to draw the arrow
THETA_KNOWN = _zone_key in ZONE_RANGE    # False only for a true UNKNOWN verdict
THETA_PRECISE = False                    # True only if a number below overrides this
THETA_RANGE = (_lo, _hi)

# refine within the zone only where we actually have a number to refine with
if VIEWING_RISK == "HIGH" and RADIO_ALPHA is not None:
    # flatter/inverted spectrum -> more beamed -> smaller angle
    THETA_ILLUSTRATIVE = float(np.clip(12.0 - (RADIO_ALPHA + 0.5) * 8.0, _lo, _hi))
    THETA_PRECISE = True
elif VIEWING_RISK == "HIGH" and OPT_POL_PCT is not None and OPT_POL_PCT > 3.0:
    THETA_ILLUSTRATIVE = float(np.clip(15.0 - OPT_POL_PCT, _lo, _hi))
    THETA_PRECISE = True
elif VIEWING_RISK == "NORMAL" and RADIO_LOUDNESS is not None and RADIO_LOUDNESS > 0:
    # more radio-loud within this band -> closer to the jet -> smaller angle
    frac = float(np.clip((np.log10(max(RADIO_LOUDNESS, 1e-3)) - 1.0) / 1.0, 0.0, 1.0))
    THETA_ILLUSTRATIVE = _hi - frac * (_hi - _lo)
    THETA_PRECISE = True
elif VIEWING_RISK == "LOW" and RADIO_LOUDNESS is not None:
    # more radio-quiet -> further from the jet -> larger angle within the zone
    frac = float(np.clip(RADIO_LOUDNESS / 10.0, 0.0, 1.0))
    THETA_ILLUSTRATIVE = _lo + (1.0 - frac) * (_hi - _lo)
    THETA_PRECISE = True


def draw_viewing_angle(target_name, risk, theta_deg, theta_known=True,
                       theta_precise=False, theta_range=None):
    """schematic of where we sit relative to the jet axis and the torus."""
    fig, ax = plt.subplots(figsize=(10.5, 7.5))
    R = 4.6

    # shaded angular zones, all measured from the jet axis (straight up).
    # plot angle = 90 - theta, so theta = 0 points along +y.
    for t1, t2, colour in [(75, 105, "#d64545"),      # within 15 deg of the axis
                           (30, 75, "#3f9142"), (105, 150, "#3f9142"),
                           (0, 30, "#d98613"), (150, 180, "#d98613")]:
        ax.add_patch(Wedge((0, 0), R, t1, t2, facecolor=colour, alpha=0.15,
                           edgecolor="none", zorder=0))

    # jet and counter-jet
    jet_half, jet_len = 12.0, 4.2
    w = jet_len * np.tan(np.radians(jet_half))
    for sign in (1, -1):
        ax.add_patch(Polygon([[0, 0], [-w, sign * jet_len], [w, sign * jet_len]],
                             closed=True, facecolor="#6fa8dc", alpha=0.9,
                             edgecolor="#2b5d8c", lw=1.5, zorder=3))

    # accretion disc, dusty torus, black hole
    ax.add_patch(Ellipse((0, 0), 3.3, 0.5, facecolor="#f5c542",
                         edgecolor="#b8860b", lw=1.4, zorder=4))
    for x in (-2.0, 2.0):
        ax.add_patch(Ellipse((x, 0), 1.8, 1.25, facecolor="#8b5a2b",
                             edgecolor="#5c3a1a", lw=1.4, alpha=0.95, zorder=5))
    ax.add_patch(Circle((0, 0), 0.23, facecolor="black", zorder=6))

    # jet axis reference line. the observer arrow is always drawn in the
    # right half, so every axis/jet label goes left to stay clear of it.
    ax.plot([0, 0], [0, R * 0.97], ls="--", color="dimgray", lw=1.3, zorder=2)
    ax.text(-0.12, R * 0.99, "jet axis", color="dimgray", fontsize=9,
            va="bottom", ha="right")

    # our line of sight
    ang = np.radians(90 - theta_deg)
    xo, yo = R * 0.97 * np.cos(ang), R * 0.97 * np.sin(ang)
    ax.annotate("", xy=(xo, yo), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", lw=2.6, color="crimson"), zorder=7)
    ax.plot([xo], [yo], marker="o", ms=13, color="crimson", zorder=8)
    ax.text(xo + 0.25, yo + 0.12, "to us\n(observer)", color="crimson",
            fontsize=10.5, fontweight="bold", ha="left", va="center")

    # the angle itself
    arc_r = 1.55
    ax.add_patch(Arc((0, 0), 2 * arc_r, 2 * arc_r, theta1=90 - theta_deg,
                     theta2=90, color="crimson", lw=2.2, zorder=7))
    if theta_precise:
        label = f"$\\theta \\approx {theta_deg:.0f}^\\circ$"
    elif theta_known and theta_range is not None:
        label = f"$\\theta$ within {theta_range[0]:.0f}$^\\circ$-{theta_range[1]:.0f}$^\\circ$"
    elif theta_known:
        label = f"$\\theta \\approx {theta_deg:.0f}^\\circ$ (typical)"
    else:
        label = "$\\theta = ?$"
    if theta_deg < 25:
        # the wedge is too narrow to hold text - put it beside the axis
        lx, ly, lha = -0.25, arc_r + 0.1, "right"
    else:
        mid = np.radians(90 - theta_deg / 2)
        lx = (arc_r + 0.35) * np.cos(mid)
        ly = (arc_r + 0.35) * np.sin(mid)
        lha = "left"
    ax.text(lx, ly, label, color="crimson", fontsize=14, fontweight="bold",
            ha=lha, va="center")

    # component labels - kept on the left, away from the observer arrow
    ax.text(-w - 0.25, jet_len * 0.72, "relativistic\njet", ha="right",
            color="#2b5d8c", fontsize=11, fontweight="bold")
    ax.annotate("accretion disc\n(what we want to measure)", xy=(-1.25, 0.12),
                xytext=(-5.4, 1.5), fontsize=10, color="#8a6d0b",
                arrowprops=dict(arrowstyle="->", color="#8a6d0b", lw=1.4))
    ax.annotate("dusty torus", xy=(-2.3, -0.45), xytext=(-5.4, -1.9),
                fontsize=10, color="#5c3a1a",
                arrowprops=dict(arrowstyle="->", color="#5c3a1a", lw=1.4))
    ax.annotate("black hole", xy=(0.15, -0.25), xytext=(1.3, -1.5),
                fontsize=10, color="black",
                arrowprops=dict(arrowstyle="->", color="black", lw=1.4))

    ax.legend(handles=[
        Patch(facecolor="#d64545", alpha=0.35,
              label=r"$\theta < 15^\circ$  jet beamed at us - DRW unreliable"),
        Patch(facecolor="#3f9142", alpha=0.35,
              label=r"$15^\circ - 60^\circ$  disc seen directly - best case"),
        Patch(facecolor="#d98613", alpha=0.35,
              label=r"$\theta > 60^\circ$  torus blocks the disc - obscured"),
    ], loc="lower right", fontsize=9.5, framealpha=0.95)

    ax.set_xlim(-6.2, 6.6)
    ax.set_ylim(-5.0, 5.6)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"{target_name} - viewing geometry     (screen verdict: {risk} risk)",
                 fontsize=13, fontweight="bold", pad=12)
    fig.text(0.5, 0.015,
             "schematic only - theta is inferred from the indicators above, not measured. "
             "measuring it properly needs VLBI jet kinematics.",
             ha="center", fontsize=9, style="italic", color="dimgray")
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


print("\n" + "=" * 70)
print(f"VIEWING ANGLE SCREEN - {AGN_NAME}")
print("=" * 70)
if VIEWING_REASONS:
    for text in VIEWING_REASONS:
        print(f"  - {text}")
else:
    print("  - nothing to go on: no classification, radio or polarization info given")
print("-" * 70)
print(f"  RISK: {VIEWING_RISK}")
print("-" * 70)

if VIEWING_RISK == "HIGH":
    print("  the jet is likely pointed near our line of sight. non-thermal jet")
    print("  emission will contaminate the optical variability, so a tau from")
    print("  this object should NOT be read as an accretion disc timescale.")
    print("  better options: pick a radio-quiet Seyfert or quasar instead, or")
    print("  keep going deliberately and report it as a contrast case.")
elif IS_OBSCURED:
    print("  this looks like an edge-on, obscured object. the torus sits between")
    print("  us and the disc, so the optical light is scattered or host-galaxy")
    print("  dominated. a DRW tau from it is not a disc timescale either -")
    print("  different reason from a blazar, same conclusion.")
elif VIEWING_RISK == "NORMAL":
    print("  a jet exists but doesn't appear to be beamed straight at us. the")
    print("  disc signal should dominate in the optical, but say so explicitly")
    print("  in the write-up and treat the tau with some caution.")
elif VIEWING_RISK == "LOW":
    print("  no sign of a beamed jet and no sign of obscuration. optical")
    print("  variability should be dominated by the accretion disc - this is")
    print("  a fair DRW target.")
else:
    print("  couldn't screen this automatically. look the object up on NED or")
    print("  SIMBAD by hand and re-run this cell before trusting the final tau.")
print("=" * 70)

draw_viewing_angle(AGN_NAME, VIEWING_RISK, THETA_ILLUSTRATIVE, THETA_KNOWN,
                   theta_precise=THETA_PRECISE, theta_range=THETA_RANGE)

print("-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  what just happened: we asked what kind of object this is and, from")
print("     that, where we are probably sitting relative to its jet. the")
print("     diagram shows that geometry.")
print()
print("  how to read the diagram: the red arrow is us. the further it sits")
print("     from the vertical jet axis, the larger theta. inside the red wedge")
print("     we are staring down the jet barrel; inside the orange wedge the")
print("     dusty torus is in the way; the green wedge in between is where")
print("     the accretion disc is visible and unbeamed, which is what we want.")
print()
print("  the angle is illustrative. it comes from the classification, not from")
print("     a measurement - never quote it as a measured value in the report.")
print()
print("  if you only entered a classification (no radio loudness, spectral")
print("     index or optical polarization), the diagram now shows a RANGE, not")
print("     a specific number - because that's genuinely all we know. two")
print("     objects with the same classification and nothing else entered will")
print("     look identical on purpose: we have equally little information about")
print("     both. enter radio loudness / alpha / polarization above if you have")
print("     it and the diagram will narrow to a specific angle within the zone.")
print()
print("  if the verdict is HIGH or the object is obscured: the pipeline will")
print("     still run, and the tau it produces will still be a real number.")
print("     it just won't be an accretion disc timescale. either swap targets")
print("     now, or carry the caveat all the way into the write-up - cell 11")
print("     does that for you automatically.")
print()
print("  if the verdict is UNKNOWN: open NED or SIMBAD, search the object,")
print("     read its classification, re-run this cell and type it in.")
print()
print("  next: cell 4, where we look at what the archive actually sent us")

## Cell 4 - Understanding the raw data

Rows, columns, every variable in the file, and what each one physically means - grouped into identification, time, photometry, quality flags and observing conditions - plus a chart of what came back.

In [ ]:
# ==============================================================
# CELL 4 - understanding the raw data before touching it
# ==============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

if "RAW" not in globals():
    raise RuntimeError("run cell 2 first - RAW isn't defined yet")

# what each ZTF column actually is: section, readable name, what it means
ZTF_COLUMNS = {
    "oid":        ("Identification", "object id",
                   "ZTF's internal id for one field/CCD-quadrant/filter combination. "
                   "the same physical source can appear under several oids"),
    "objectid":   ("Identification", "object id",
                   "alternate object identifier returned by some queries"),
    "field":      ("Identification", "field number",
                   "which ZTF survey field this exposure belongs to"),
    "ccdid":      ("Identification", "CCD id",
                   "which of the 16 CCDs in the camera recorded it"),
    "qid":        ("Identification", "quadrant id",
                   "which quadrant (1-4) of that CCD"),
    "fid":        ("Identification", "filter id",
                   "numeric filter code: 1 = g, 2 = r, 3 = i"),
    "filtercode": ("Identification", "filter name",
                   "text filter code: zg / zr / zi. different filters trace "
                   "different disc radii, so never mix them in one fit"),
    "programid":  ("Identification", "survey program",
                   "1 = public survey, 2 = collaboration, 3 = Caltech time"),

    "hjd":        ("Time / epoch", "heliocentric Julian date",
                   "epoch corrected to the Sun's frame, in days"),
    "mjd":        ("Time / epoch", "modified Julian date",
                   "THE time axis for this project. a continuous day count since "
                   "17 Nov 1858, so time differences are plain subtraction"),
    "expid":      ("Time / epoch", "exposure id",
                   "unique id of the individual exposure this point came from"),
    "exptime":    ("Time / epoch", "exposure time",
                   "how long the shutter was open, in seconds (usually 30)"),

    "mag":        ("Photometry", "magnitude",
                   "calibrated PSF-fit brightness. RUNS BACKWARDS - a smaller "
                   "number means a brighter object. this is what we fit"),
    "magerr":     ("Photometry", "magnitude uncertainty",
                   "1-sigma error on mag from photon counting + instrument noise. "
                   "this is what weights every point in the DRW likelihood"),
    "limitmag":   ("Photometry", "limiting magnitude",
                   "faintest 5-sigma detection possible in that exposure - a "
                   "measure of how good conditions were"),
    "clrcoeff":   ("Photometry", "colour coefficient",
                   "colour term used in the photometric calibration of that night"),
    "magzp":      ("Photometry", "zero point",
                   "photometric zero point of the exposure"),
    "magzprms":   ("Photometry", "zero point scatter",
                   "rms scatter of the zero point calibration"),

    "catflags":   ("Quality flags", "quality bitmask",
                   "bitmask of known problems (cosmic rays, bad pixels, clouds, "
                   "bad seeing). bit 15 (value 32768) is the bad-quality bit we cut on"),
    "chi":        ("Quality flags", "PSF fit chi",
                   "goodness of the point-spread-function fit. near 1 means a "
                   "clean point source; large values mean a blended or bad fit"),
    "sharp":      ("Quality flags", "PSF sharpness",
                   "how pointlike the source looks. near 0 = point source, "
                   "negative = too sharp (cosmic ray), positive = extended"),

    "ra":         ("Astrometry & conditions", "right ascension",
                   "measured RA of the source in that exposure, in degrees"),
    "dec":        ("Astrometry & conditions", "declination",
                   "measured Dec of the source in that exposure, in degrees"),
    "airmass":    ("Astrometry & conditions", "airmass",
                   "how much atmosphere we looked through. 1.0 = straight up, "
                   "higher = closer to the horizon and noisier"),
    "seeing":     ("Astrometry & conditions", "seeing",
                   "atmospheric blurring in arcsec during that exposure"),
}

SECTION_ORDER = ["Identification", "Time / epoch", "Photometry",
                 "Quality flags", "Astrometry & conditions", "Not in our dictionary"]

print("=" * 70)
print(f"RAW DATA STRUCTURE - {AGN_NAME}")
print("=" * 70)
print(f"rows (epochs returned) : {RAW.shape[0]}")
print(f"columns (variables)    : {RAW.shape[1]}")
print(f"\nall variables present  : {', '.join(RAW.columns.tolist())}")
print("=" * 70)

rows = []
for col in RAW.columns:
    section, nice, detail = ZTF_COLUMNS.get(
        col, ("Not in our dictionary", col, "not in our column dictionary - "
              "check the IRSA ZTF documentation for what this one means")
    )
    series = RAW[col]
    n_missing = int(series.isna().sum())
    if pd.api.types.is_numeric_dtype(series) and series.notna().any():
        rng = f"{series.min():.5g} to {series.max():.5g}"
    else:
        uniq = series.dropna().unique()
        rng = ", ".join(map(str, uniq[:4])) + (" ..." if len(uniq) > 4 else "")
    rows.append({
        "section": section, "column": col, "means": nice,
        "dtype": str(series.dtype), "missing": n_missing,
        "range / values": rng, "detail": detail,
    })

INFO = pd.DataFrame(rows)

for section in SECTION_ORDER:
    part = INFO[INFO["section"] == section]
    if part.empty:
        continue
    print(f"\n{'=' * 70}\n{section.upper()}\n{'=' * 70}")
    for _, r in part.iterrows():
        print(f"\n  {r['column']}  ({r['means']})")
        print(f"     dtype      : {r['dtype']}")
        print(f"     missing    : {r['missing']}")
        print(f"     values     : {r['range / values']}")
        print(f"     what it is : {r['detail']}")

print("\n\nsame thing as a table:")
display(INFO[["section", "column", "means", "dtype", "missing", "range / values"]])

# quick visual: how complete is each column, and what bands came back
fig, axes = plt.subplots(1, 2, figsize=(13, max(4, 0.32 * len(RAW.columns))))

completeness = (RAW.notna().sum() / len(RAW) * 100).sort_values()
axes[0].barh(completeness.index, completeness.values, color="steelblue")
axes[0].set_xlabel("% of rows with a value")
axes[0].set_xlim(0, 105)
axes[0].set_title("completeness per variable")
axes[0].grid(axis="x", alpha=0.3)

if "filtercode" in RAW.columns:
    counts = RAW["filtercode"].value_counts()
    axes[1].bar(counts.index.astype(str), counts.values, color="indianred")
    axes[1].set_ylabel("epochs")
    axes[1].set_title("epochs per filter band")
    for i, v in enumerate(counts.values):
        axes[1].text(i, v, str(v), ha="center", va="bottom")
else:
    axes[1].axis("off")
    axes[1].set_title("no filtercode column returned")

plt.tight_layout()
plt.show()

print("\n" + "-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  why this cell exists: this is the step people skip and regret. you")
print("     are looking at an instrument's output, not a tidy dataset. before")
print("     cleaning anything you should be able to say, in your own words,")
print("     what every column physically represents.")
print()
print("  the three that matter for us: mjd (time), mag (brightness) and")
print("     magerr (how much to trust each brightness). everything else is")
print("     either an id, a quality check, or observing conditions.")
print()
print("  the trap everyone hits once: magnitudes run BACKWARDS. mag 14 is")
print("     brighter than mag 15. that's why every light curve plot in")
print("     astronomy has a flipped y-axis.")
print()
print("  catflags is a bitmask, not a number. it packs several yes/no")
print("     answers into one integer. we test one specific bit (32768 =")
print("     bad quality) rather than asking whether the whole value is zero,")
print("     because a row can carry a harmless flag and still be good data.")
print()
print("  if a column says 'not in our dictionary': that's fine, it just means")
print("     we didn't write a description for it. look it up in the IRSA ZTF")
print("     documentation if you need it.")
print()
print("  deliverable for your report: write one sentence per column in your")
print("     own words. if you can't write the sentence, you don't understand")
print("     the column yet.")
print()
print("  next: cell 5, where you choose which variables to carry forward")
print("-" * 70)

## Cell 5 - Choose your variables and check them

Pick what you need, then each one is checked against the quality flags and rated: usable, usable with cuts, or high risk.

In [ ]:
# ==============================================================
# CELL 5 - choose the variables, then check whether they're usable
# ==============================================================
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

if "RAW" not in globals():
    raise RuntimeError("run cell 2 first - RAW isn't defined yet")

BAD_QUALITY_BIT = 32768   # ZTF catflags bit 15

print("variables available in this file:\n")
cols = RAW.columns.tolist()
for i, c in enumerate(cols, start=1):
    print(f"  {i:>2}. {c}")

print("\nthe DRW fit needs mjd, mag and magerr at minimum.")
choice = input("\nwhich variables do you want? (comma separated names or numbers, "
               "Enter for mjd,mag,magerr): ").strip()

if choice == "":
    CHOSEN = [c for c in ("mjd", "mag", "magerr") if c in cols]
else:
    CHOSEN = []
    for token in choice.split(","):
        token = token.strip()
        if not token:
            continue
        if token.isdigit() and 1 <= int(token) <= len(cols):
            CHOSEN.append(cols[int(token) - 1])
        elif token in cols:
            CHOSEN.append(token)
        else:
            print(f"   -> '{token}' isn't a column in this file, skipping it")
    CHOSEN = list(dict.fromkeys(CHOSEN))   # drop duplicates, keep order

if not CHOSEN:
    raise RuntimeError("no valid variables picked - re-run this cell")

missing_essential = [c for c in ("mjd", "mag", "magerr") if c not in CHOSEN]
print(f"\nchosen: {', '.join(CHOSEN)}")
if missing_essential:
    print(f"heads up: {', '.join(missing_essential)} not selected - cells 6-8 need "
          f"these, they'll be pulled in automatically for the fit")

# how much of the file is flagged bad, for comparison below
if "catflags" in RAW.columns:
    flags = RAW["catflags"].fillna(0).astype(int)
    bad_mask = (flags & BAD_QUALITY_BIT) != 0
    n_bad = int(bad_mask.sum())
    print(f"\nquality flags: {n_bad} of {len(RAW)} rows "
          f"({100 * n_bad / len(RAW):.1f}%) have the bad-quality bit set")
else:
    bad_mask = pd.Series(False, index=RAW.index)
    print("\nno catflags column in this file - can't cross-check against flags")

report = []
for col in CHOSEN:
    s = RAW[col]
    missing_pct = 100 * s.isna().mean()
    notes = []
    verdict = "USABLE - low risk"

    if missing_pct > 20:
        verdict = "HIGH RISK"
        notes.append(f"{missing_pct:.0f}% of rows are empty")
    elif missing_pct > 0:
        notes.append(f"{missing_pct:.1f}% missing")

    if col == "mag":
        med_err = float(RAW["magerr"].median()) if "magerr" in RAW.columns else np.nan
        spread = float(s.std())
        med = float(s.median())
        mad = 1.4826 * float(np.median(np.abs(s.dropna() - med))) if s.notna().any() else 0.0
        n_out = int((np.abs(s - med) > 5 * mad).sum()) if mad > 0 else 0
        notes.append(f"median = {med:.3f}, scatter = {spread:.3f} mag")
        if np.isfinite(med_err):
            notes.append(f"typical error = {med_err:.3f} mag")
            if spread <= med_err:
                verdict = "HIGH RISK"
                notes.append("scatter is no bigger than the error bars - "
                             "no variability detected above noise")
            elif spread < 2 * med_err:
                verdict = "USABLE WITH CUTS - some noise"
                notes.append("variability is only marginally above the noise")
        if n_out:
            notes.append(f"{n_out} points beyond 5x MAD (outliers to clip in cell 6)")
            if verdict == "USABLE - low risk" and n_out > 0.05 * len(s):
                verdict = "USABLE WITH CUTS - some noise"

    elif col == "magerr":
        med_err = float(s.median())
        p90 = float(s.quantile(0.90))
        notes.append(f"median = {med_err:.3f}, 90th percentile = {p90:.3f} mag")
        if med_err > 0.1:
            verdict = "HIGH RISK"
            notes.append("typical error above 0.1 mag - this source is faint "
                         "for ZTF, the DRW fit will struggle")
        elif med_err > 0.05:
            verdict = "USABLE WITH CUTS - some noise"
            notes.append("errors are on the large side, keep the magerr cut tight")

    elif col == "catflags":
        notes.append(f"{100 * (flags != 0).mean():.1f}% of rows carry some flag, "
                     f"{100 * bad_mask.mean():.1f}% carry the bad-quality bit")
        verdict = "USABLE - low risk (this is the cut, not the data)"

    elif col == "mjd":
        span = float(s.max() - s.min())
        notes.append(f"baseline = {span:.0f} days ({span / 365.25:.2f} years)")
        if span < 365:
            verdict = "HIGH RISK"
            notes.append("under a year of data - too short to measure a timescale")
        elif span < 3 * 365:
            verdict = "USABLE WITH CUTS - some noise"
            notes.append("short baseline, expect a wide confidence interval")

    elif col == "chi":
        notes.append(f"median = {float(s.median()):.2f} (near 1 means clean PSF fits)")

    elif col in ("ra", "dec"):
        notes.append(f"scatter = {float(s.std()) * 3600:.2f} arcsec around the mean position")

    # compare flagged vs unflagged rows for this variable
    if bad_mask.any() and pd.api.types.is_numeric_dtype(s):
        clean_mean = float(s[~bad_mask].mean())
        bad_mean = float(s[bad_mask].mean())
        if np.isfinite(clean_mean) and np.isfinite(bad_mean) and clean_mean != 0:
            shift = 100 * abs(bad_mean - clean_mean) / abs(clean_mean)
            if shift > 1:
                notes.append(f"flagged rows sit {shift:.1f}% away from unflagged rows "
                             f"- worth cutting them")

    report.append({
        "variable": col,
        "missing %": round(missing_pct, 2),
        "verdict": verdict,
        "what we found": "; ".join(notes) if notes else "nothing unusual",
    })

QUALITY_REPORT = pd.DataFrame(report)

print("\n" + "=" * 70)
print("VARIABLE QUALITY CHECK")
print("=" * 70)
for r in report:
    print(f"\n  {r['variable']}")
    print(f"     verdict : {r['verdict']}")
    print(f"     details : {r['what we found']}")
print("=" * 70)
display(QUALITY_REPORT)

if any("HIGH RISK" in r["verdict"] for r in report):
    print("\nsomething came back HIGH RISK above. read why before continuing -")
    print("cleaning in cell 6 fixes noise, it does not fix a source that simply")
    print("isn't varying above its error bars.")
else:
    print("\nnothing alarming here")

print("\n" + "-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  just press Enter unless you have a reason not to. mjd, mag and magerr")
print("     are what the fit needs; the rest are useful for understanding the")
print("     data but don't feed the model.")
print()
print("  what the verdicts mean:")
print("     USABLE - low risk        nothing wrong, carry on")
print("     USABLE WITH CUTS         real but noisy - cleaning in cell 6 will")
print("                              help, keep the caveat in mind")
print("     HIGH RISK                a problem cleaning cannot fix")
print()
print("  the one that ends the analysis: if mag comes back HIGH RISK because")
print("     the scatter is no bigger than the error bars, this source is not")
print("     measurably varying. no amount of cleaning creates a signal that")
print("     isn't there - pick a brighter or more active target.")
print()
print("  why we compare against the flags: if the flagged rows sit at a")
print("     noticeably different brightness than the unflagged ones, the flags")
print("     are catching something real and cutting them will matter.")
print()
print("  next: cell 6, cleaning")
print("-" * 70)

## Cell 6 - Clean the data

Picks the right object if several came back, keeps one filter band, applies the four cleaning steps, and gives a verdict on whether what's left is actually good enough to plot and fit.

In [ ]:
# ==============================================================
# CELL 6 - clean the data, then say whether it's fit to plot
# ==============================================================
import numpy as np
import pandas as pd

if "RAW" not in globals():
    raise RuntimeError("run cell 2 first - RAW isn't defined yet")

BAD_QUALITY_BIT = 32768
MAX_MAGERR = 0.2      # drop points with errors bigger than this
MAD_NSIGMA = 5.0      # sigma-clip threshold, in robust MAD units

steps = []
df = RAW.copy()
steps.append(("raw rows returned by the archive", len(df)))

# --- pick the right object if several oids came back -------------------
if "oid" in df.columns and df["oid"].nunique() > 1:
    cand = []
    for oid, g in df.groupby("oid"):
        if "ra" in g.columns and "dec" in g.columns:
            sep = np.sqrt(
                ((g["ra"].mean() - RA_DEG) * np.cos(np.radians(DEC_DEG))) ** 2
                + (g["dec"].mean() - DEC_DEG) ** 2
            ) * 3600.0
        else:
            sep = np.nan
        cand.append({"oid": oid, "n_points": len(g), "sep_arcsec": sep})
    # sort by number of points FIRST, separation only breaks ties. sorting
    # by separation first lets a sparse spurious match outrank the real
    # source just for being a fraction of an arcsec closer.
    cand = pd.DataFrame(cand).sort_values(["n_points", "sep_arcsec"],
                                          ascending=[False, True])
    print("several ZTF object ids came back for this position:\n")
    print(cand.to_string(index=False))
    TARGET_OID = cand.iloc[0]["oid"]
    print(f"\nusing oid {TARGET_OID} - most points, and closest of those")
    df = df[df["oid"] == TARGET_OID].copy()
    steps.append((f"rows belonging to oid {TARGET_OID}", len(df)))
else:
    TARGET_OID = df["oid"].iloc[0] if "oid" in df.columns else "n/a"

# --- keep a single filter band -----------------------------------------
if "filtercode" in df.columns and df["filtercode"].nunique() > 1:
    bands_present = sorted(df["filtercode"].dropna().unique())
    keep = f"z{BAND}"
    print(f"\nmore than one band came back ({', '.join(bands_present)}) - keeping {keep} only.")
    print("different filters trace different disc radii, mixing them would "
          "smear the timescale.")
    df = df[df["filtercode"] == keep].copy()
    steps.append((f"rows in band {keep}", len(df)))

# --- the four cleaning steps -------------------------------------------
df = df.dropna(subset=["mjd", "mag", "magerr"])
steps.append(("after dropping rows with missing mjd/mag/magerr", len(df)))

if "catflags" in df.columns:
    df = df[(df["catflags"].fillna(0).astype(int) & BAD_QUALITY_BIT) == 0]
    steps.append(("after cutting the catflags bad-quality bit", len(df)))

df = df[df["magerr"] < MAX_MAGERR]
steps.append((f"after cutting magerr >= {MAX_MAGERR}", len(df)))

if df.empty:
    raise RuntimeError("nothing survived cleaning - loosen MAX_MAGERR or check the target")

med = df["mag"].median()
mad = 1.4826 * np.median(np.abs(df["mag"] - med))
if mad > 0:
    df = df[np.abs(df["mag"] - med) < MAD_NSIGMA * mad]
steps.append((f"after {MAD_NSIGMA}-sigma MAD clip against the median", len(df)))

CLEAN = df.sort_values("mjd").reset_index(drop=True)

# --- collapse same-night repeats ---------------------------------------
night = np.floor(CLEAN["mjd"].to_numpy()).astype(int)
mjd_a, mag_a, err_a = (CLEAN["mjd"].to_numpy(), CLEAN["mag"].to_numpy(),
                       CLEAN["magerr"].to_numpy())
out = {"mjd": [], "mag": [], "magerr": []}
for n in np.unique(night):
    m = night == n
    w = 1.0 / err_a[m] ** 2
    out["mjd"].append(np.sum(w * mjd_a[m]) / np.sum(w))
    out["mag"].append(np.sum(w * mag_a[m]) / np.sum(w))
    out["magerr"].append(1.0 / np.sqrt(np.sum(w)))
NIGHTLY = pd.DataFrame(out).sort_values("mjd").reset_index(drop=True)
steps.append(("after binning same-night repeats into one point each", len(NIGHTLY)))

print("\n" + "=" * 70)
print("CLEANING TRAIL")
print("=" * 70)
for label, n in steps:
    print(f"  {n:>6}   {label}")
print("=" * 70)

# --- is this good enough to plot and fit? ------------------------------
n_points = len(NIGHTLY)
baseline = float(NIGHTLY["mjd"].max() - NIGHTLY["mjd"].min())
med_err = float(NIGHTLY["magerr"].median())
scatter = float(NIGHTLY["mag"].std())
# excess variance: how much scatter is left once measurement noise is removed
excess_var = scatter ** 2 - float(np.mean(NIGHTLY["magerr"] ** 2))
excess_rms = float(np.sqrt(excess_var)) if excess_var > 0 else 0.0
kept_pct = 100 * len(CLEAN) / len(RAW)

problems, warnings_ = [], []
if n_points < 50:
    problems.append(f"only {n_points} nightly points - too few to fit a timescale")
elif n_points < 100:
    warnings_.append(f"{n_points} nightly points is on the thin side")

if baseline < 365:
    problems.append(f"baseline is only {baseline:.0f} days - under a year")
elif baseline < 3 * 365:
    warnings_.append(f"baseline {baseline / 365.25:.1f} years - short, expect a wide CI")

if excess_rms <= 0 or excess_rms < 0.3 * med_err:
    # intrinsic scatter this far below the error bars can't be distinguished
    # from noise, so there is no variability signal to fit a timescale to
    problems.append(f"intrinsic variability ({excess_rms:.4f} mag) is not "
                    f"measurably above the noise ({med_err:.4f} mag typical "
                    f"error) - this source looks flat within its error bars")
elif excess_rms < med_err:
    warnings_.append(f"intrinsic variability ({excess_rms:.3f} mag) is smaller than "
                     f"the typical error bar ({med_err:.3f} mag)")

if kept_pct < 40:
    warnings_.append(f"only {kept_pct:.0f}% of the raw rows survived cleaning")

print(f"\n  nightly points        : {n_points}")
print(f"  baseline              : {baseline:.0f} days ({baseline / 365.25:.2f} years)")
print(f"  typical error bar     : {med_err:.4f} mag")
print(f"  observed scatter      : {scatter:.4f} mag")
print(f"  intrinsic variability : {excess_rms:.4f} mag (noise removed)")
print(f"  kept from raw         : {kept_pct:.1f}%")

print("\n" + "=" * 70)
if problems:
    DATA_VERDICT = "NOT GOOD FOR PLOTTING / FITTING"
    print(f"VERDICT: {DATA_VERDICT}")
    for p in problems:
        print(f"  - {p}")
    print("\n  plotting it is still fine to look at, but don't report a tau from")
    print("  this - fix the input first (different band, wider radius, or a")
    print("  brighter target).")
elif warnings_:
    DATA_VERDICT = "USABLE WITH CAVEATS"
    print(f"VERDICT: {DATA_VERDICT}")
    for w in warnings_:
        print(f"  - {w}")
    print("\n  good enough to plot and fit, but carry these caveats into the write-up.")
else:
    DATA_VERDICT = "GOOD FOR PLOTTING AND FITTING"
    print(f"VERDICT: {DATA_VERDICT}")
    print("  enough points, long enough baseline, and real variability above")
    print("  the noise. go ahead.")
print("=" * 70)

print("\n" + "-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  cleaning is not cheating. every one of these cuts removes points we")
print("     have a stated reason to distrust, decided before looking at the")
print("     answer. what would be cheating is dropping points because they")
print("     spoil a result you wanted.")
print()
print("  the four steps, and why:")
print("     missing values   - nothing to fit")
print("     catflags bit     - the pipeline itself flagged these as bad")
print("     magerr cut       - a point with a huge error bar tells us little")
print("     MAD sigma clip   - removes outliers using a robust spread. plain")
print("                        standard deviation gets dragged around by the")
print("                        very outliers you're trying to catch; the median")
print("                        absolute deviation doesn't")
print("     nightly binning  - two exposures on the same night are one sample")
print("                        of a months-long process, not two")
print()
print("  intrinsic variability is the number to watch. it's the scatter left")
print("     after removing what the error bars alone would produce. if it's")
print("     near zero, the source is flat and there is no timescale to measure.")
print()
print("  if the verdict is NOT GOOD: don't just push on. try the other band,")
print("     widen the search radius slightly, or pick a brighter and more")
print("     active target. a tau fitted to noise is still a number, and it is")
print("     still meaningless.")
print()
print("  record the before/after counts above for your report - that trail is")
print("     what makes the analysis reproducible.")
print()
print("  next: cell 7, look at the light curve with your own eyes")
print("-" * 70)

## Cell 7 - Plot the light curve

Raw, cleaned and nightly-binned on one axis, plus the sampling pattern underneath so the seasonal gaps are visible. Look at this before trusting any statistics.

In [ ]:
# ==============================================================
# CELL 7 - plot the light curve
# ==============================================================
import os

import numpy as np
import matplotlib.pyplot as plt

if "NIGHTLY" not in globals():
    raise RuntimeError("run cell 6 first - NIGHTLY isn't defined yet")

fig, axes = plt.subplots(2, 1, figsize=(13, 9),
                         gridspec_kw={"height_ratios": [2, 1]})

ax = axes[0]
ax.errorbar(RAW["mjd"], RAW["mag"], yerr=RAW.get("magerr"),
            fmt=".", color="lightgray", alpha=0.45, markersize=4,
            label=f"raw, all rows returned (N={len(RAW)})", zorder=1)
ax.errorbar(CLEAN["mjd"], CLEAN["mag"], yerr=CLEAN["magerr"],
            fmt=".", color="steelblue", alpha=0.5, markersize=4,
            label=f"passed quality cuts (N={len(CLEAN)})", zorder=2)
ax.errorbar(NIGHTLY["mjd"], NIGHTLY["mag"], yerr=NIGHTLY["magerr"],
            fmt="o", color="crimson", markersize=4, elinewidth=0.9,
            label=f"nightly binned, used in the fit (N={len(NIGHTLY)})", zorder=3)

ax.invert_yaxis()   # magnitudes run backwards - brighter goes up
ax.set_ylabel(f"{BAND}-band magnitude")
ax.set_title(f"{AGN_NAME} - ZTF {BAND}-band light curve "
             f"({baseline / 365.25:.1f} year baseline)")
ax.legend(loc="best")
ax.grid(alpha=0.3)

# where the seasonal gaps are - a sanity check on the cadence
gaps = np.diff(NIGHTLY["mjd"].to_numpy())
mid = NIGHTLY["mjd"].to_numpy()[:-1] + gaps / 2
ax2 = axes[1]
ax2.plot(mid, gaps, ".", color="darkslategray", markersize=4)
ax2.axhline(30, color="orange", linestyle="--", alpha=0.7, label="30 day gap")
ax2.axhline(100, color="red", linestyle="--", alpha=0.7, label="100 day gap (seasonal)")
ax2.set_yscale("log")
ax2.set_xlabel("MJD")
ax2.set_ylabel("gap to next point (days)")
ax2.set_title("sampling - the tall spikes are the annual seasonal gaps")
ax2.legend(loc="best")
ax2.grid(alpha=0.3)

plt.tight_layout()

LC_PLOT_PATH = os.path.join(SAVE_DIR, f"{_safe}_lightcurve.png")
plt.savefig(LC_PLOT_PATH, dpi=200)
plt.show()

print(f"saved to {LC_PLOT_PATH}\n")
print(f"  median gap between nights : {np.median(gaps):.1f} days")
print(f"  longest gap               : {gaps.max():.0f} days")
print(f"  brightest point           : {NIGHTLY['mag'].min():.3f} mag")
print(f"  faintest point            : {NIGHTLY['mag'].max():.3f} mag")
print(f"  peak-to-peak variation    : {NIGHTLY['mag'].max() - NIGHTLY['mag'].min():.3f} mag")
print("\n" + "-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  this is the most important plot in the project. every statistic that")
print("     follows is a summary of this picture - if the picture is wrong, the")
print("     statistics are confidently wrong.")
print()
print("  the y-axis is upside down on purpose. magnitudes run backwards, so")
print("     flipping the axis puts 'brighter' at the top where you expect it.")
print()
print("  what to look for in the top panel:")
print("     - does the brightness actually wander, or is it a flat line with")
print("       error bars? wandering is the signal we're about to fit")
print("     - are there isolated points far from everything else that survived")
print("       cleaning? if so, tighten the cuts in cell 6 and re-run")
print("     - do the grey raw points that got removed look like genuine junk?")
print()
print("  what the bottom panel tells you: the tall spikes are the annual gaps")
print("     when the target is behind the Sun and unobservable. they are")
print("     normal and expected, but they're exactly why a timescale longer")
print("     than a season is hard to pin down - the fit has to bridge them.")
print()
print("  rule of thumb you'll meet again in cell 8: you cannot reliably")
print("     measure a timescale longer than about a tenth of your baseline.")
print()
print("  put this figure in your report. it's saved at full resolution above.")
print()
print("  next: cell 8, fit the timescale")
print("-" * 70)

## Cell 8 - Fit the damping timescale

Fits the damped random walk model and reports tau, sigma and a profile likelihood confidence interval, as a table.

In [ ]:
# ==============================================================
# CELL 8 - fit the damping timescale (DRW)
# ==============================================================
#
# The fit looks for the tau and sigma that make the observed run of ups
# and downs most likely, assuming brightness does a damped random walk:
# wanders at random, but gets pulled back towards a mean over a
# characteristic time tau.
#
# Kernel convention used here (celerite / MacLeod et al. 2010):
#       k(dt) = sigma^2 * exp(-dt / tau)
# NOT the Kelly et al. (2009) form S(dt) = (tau*sigma^2/2) exp(-dt/tau).
# Same shape, different sigma - don't mix numbers from the two.

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from celerite2 import terms, GaussianProcess

try:
    from IPython.display import display
except Exception:
    display = print

if "NIGHTLY" not in globals():
    raise RuntimeError("run cell 6 first - NIGHTLY isn't defined yet")

T = NIGHTLY["mjd"].to_numpy()
Y = NIGHTLY["mag"].to_numpy()
YERR = NIGHTLY["magerr"].to_numpy()
BASELINE = float(T.max() - T.min())

TAU0_GRID = [10, 100, 500, 1000, 3000, 8000]      # starting guesses, days
SIGMA0_GRID = [0.05, 0.15, 0.3]                   # starting guesses, mag


def drw_nll(params, t, y, yerr):
    mean, l_sigma, l_tau = params
    sigma, tau = np.exp(l_sigma), np.exp(l_tau)
    if not np.isfinite(tau) or tau <= 0 or not np.isfinite(sigma):
        return 1e10
    gp = GaussianProcess(terms.RealTerm(a=sigma ** 2, c=1.0 / tau), mean=mean)
    try:
        gp.compute(t, diag=yerr ** 2, quiet=True)
    except Exception:
        return 1e10
    ll = gp.log_likelihood(y)
    return -ll if np.isfinite(ll) else 1e10


def fit_drw(t, y, yerr, tau0_grid=TAU0_GRID, sigma0_grid=SIGMA0_GRID):
    """multi-start fit - a single start can settle in a local minimum."""
    mean0 = float(np.mean(y))
    best = None
    for tau0 in tau0_grid:
        for sigma0 in sigma0_grid:
            res = minimize(drw_nll, [mean0, np.log(sigma0), np.log(tau0)],
                           args=(t, y, yerr), method="L-BFGS-B")
            if res.success and np.isfinite(res.fun) and (best is None or res.fun < best.fun):
                best = res
    if best is None:
        raise RuntimeError("the fit didn't converge from any starting point")
    mean_fit, l_sigma_fit, l_tau_fit = best.x
    return {"mean": float(mean_fit), "sigma": float(np.exp(l_sigma_fit)),
            "tau": float(np.exp(l_tau_fit)), "nll": float(best.fun)}


def profile_tau_ci(t, y, yerr, fit, delta=1.92):
    """95% interval on tau: step tau across a grid, refit everything else
    at each step, keep the range where the likelihood stays within 1.92
    of its best value."""
    grid = np.geomspace(5, 20 * (t.max() - t.min()), 60)
    ll_best = -fit["nll"]
    x0 = [fit["mean"], np.log(fit["sigma"])]
    prof = []
    for tau in grid:
        def nll_fixed(p, tau=tau):
            return drw_nll([p[0], p[1], np.log(tau)], t, y, yerr)
        res = minimize(nll_fixed, x0, method="L-BFGS-B")
        prof.append((tau, -res.fun))
    prof = pd.DataFrame(prof, columns=["tau", "loglike"])
    inside = prof[(ll_best - prof["loglike"]) < delta]
    if inside.empty:
        return None, None, prof
    return float(inside["tau"].min()), float(inside["tau"].max()), prof


print(f"fitting {len(T)} nightly points over {BASELINE:.0f} days ...")
print(f"trying {len(TAU0_GRID) * len(SIGMA0_GRID)} starting points\n")

FIT = fit_drw(T, Y, YERR)
TAU_CI_LO, TAU_CI_HI, TAU_PROFILE = profile_tau_ci(T, Y, YERR, FIT)
TAU_OVER_BASELINE = FIT["tau"] / BASELINE

FIT_TABLE = pd.DataFrame([
    ("Target", AGN_NAME),
    ("ZTF object id", TARGET_OID),
    ("Band", BAND),
    ("Nightly points fitted", len(T)),
    ("Baseline (days)", f"{BASELINE:.2f}"),
    ("Baseline (years)", f"{BASELINE / 365.25:.2f}"),
    ("Mean magnitude", f"{FIT['mean']:.4f}"),
    ("DRW sigma (mag)", f"{FIT['sigma']:.4f}"),
    ("DRW tau (days)", f"{FIT['tau']:.2f}"),
    ("DRW tau (years)", f"{FIT['tau'] / 365.25:.2f}"),
    ("tau / baseline", f"{TAU_OVER_BASELINE:.4f}"),
    ("95% CI on tau (days)",
     f"{TAU_CI_LO:.1f} - {TAU_CI_HI:.1f}" if TAU_CI_LO is not None else "unconstrained"),
    ("-log likelihood", f"{FIT['nll']:.3f}"),
], columns=["quantity", "value"])

print("=" * 70)
print("DRW FIT RESULT")
print("=" * 70)
for _, r in FIT_TABLE.iterrows():
    print(f"  {r['quantity']:<24}: {r['value']}")
print("=" * 70)
display(FIT_TABLE)

print(f"\nin words: the brightness of {AGN_NAME} wanders and gets pulled back")
print(f"towards its mean on a timescale of about {FIT['tau']:.0f} days")
print(f"({FIT['tau'] / 365.25:.1f} years), with an amplitude of {FIT['sigma']:.3f} mag.")

if TAU_OVER_BASELINE > 0.1:
    print(f"\nWARNING: tau is {TAU_OVER_BASELINE:.2f} of the baseline. above ~0.1 the")
    print("fit systematically underestimates tau and the number is unreliable on")
    print("its own - cell 10 measures how badly, for this exact cadence.")
else:
    print(f"\ntau/baseline = {TAU_OVER_BASELINE:.3f}, comfortably under the 0.1 rule of")
    print("thumb, so the fit is in the regime where it can be trusted.")

print("\n" + "-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  what tau means in plain words: leave the source alone for a time much")
print("     shorter than tau and it barely changes. wait much longer than tau")
print("     and it has forgotten where it started. tau is the memory of the")
print("     accretion disc, and it's tied to the physical conditions there.")
print()
print("  what sigma means: how far the brightness typically strays from its")
print("     long-term average, in magnitudes. amplitude, not speed.")
print()
print("  why we start the fit from 18 different places: the likelihood surface")
print("     has local dips. a single starting guess can settle into one and")
print("     report it confidently. trying a spread of starting points and")
print("     keeping the best result guards against that.")
print()
print("  the confidence interval is not decoration. a maximum-likelihood tau")
print("     on its own always looks more precise than it is. if the interval")
print("     spans a factor of several, say so in the report - that IS the")
print("     result, not a footnote to it.")
print()
print("  about the sigma convention: this notebook uses k(dt) = sigma^2")
print("     exp(-dt/tau), which is what celerite2 and MacLeod et al. (2010)")
print("     use. some papers (Kelly et al. 2009) write the covariance as")
print("     (tau*sigma^2/2) exp(-dt/tau) instead. same model, different sigma.")
print("     if you compare your sigma with a published one, check which")
print("     convention that paper used first or you'll compare two different")
print("     quantities and think you disagree.")
print()
print("  the 0.1 rule: if tau is more than about a tenth of your baseline, the")
print("     fit systematically underestimates it. cell 10 measures exactly how")
print("     badly for this particular light curve instead of guessing.")
print()
print("  next: cell 9, gather everything and check it against published work")
print("-" * 70)

## Cell 9 - All results, and the check against past work

Everything in one place, internal consistency checks, and a prompt to compare against a published value.

In [ ]:
# ==============================================================
# CELL 9 - all results so far, and a check against past work
# ==============================================================
import numpy as np

if "FIT" not in globals():
    raise RuntimeError("run cell 8 first - FIT isn't defined yet")

print("=" * 70)
print(f"EVERYTHING WE HAVE FOR {AGN_NAME} SO FAR")
print("=" * 70)
print(f"  analysis date          : {OBS_DATE}")
print(f"  position               : RA {RA_DEG}, Dec {DEC_DEG} "
      f"(radius {RADIUS_DEG} deg)")
print(f"  archive / band         : ZTF {BAND}-band, oid {TARGET_OID}")
print(f"  viewing angle risk     : {VIEWING_RISK}")
print(f"  raw rows               : {len(RAW)}")
print(f"  passed quality cuts    : {len(CLEAN)}")
print(f"  nightly points fitted  : {len(NIGHTLY)}")
print(f"  baseline               : {BASELINE:.0f} days ({BASELINE / 365.25:.2f} years)")
print(f"  data verdict           : {DATA_VERDICT}")
print(f"  mean magnitude         : {FIT['mean']:.4f}")
print(f"  DRW sigma              : {FIT['sigma']:.4f} mag")
print(f"  DRW tau                : {FIT['tau']:.1f} days ({FIT['tau'] / 365.25:.2f} yr)")
print(f"  95% CI on tau          : "
      f"{f'{TAU_CI_LO:.1f} - {TAU_CI_HI:.1f} days' if TAU_CI_LO else 'unconstrained'}")
print(f"  tau / baseline         : {TAU_OVER_BASELINE:.4f}")
print("=" * 70)

# --- internal consistency checks before comparing outward --------------
print("\nINTERNAL CHECKS")
internal = []
if TAU_OVER_BASELINE > 0.1:
    internal.append(("FAIL", f"tau/baseline = {TAU_OVER_BASELINE:.2f} exceeds 0.1 - "
                             "known to bias tau low, needs cell 10"))
else:
    internal.append(("PASS", f"tau/baseline = {TAU_OVER_BASELINE:.3f} is under 0.1"))

if TAU_CI_LO is None:
    internal.append(("FAIL", "the 95% interval is unconstrained - the data can't "
                             "pin tau down at all"))
elif TAU_CI_HI / max(TAU_CI_LO, 1e-9) > 10:
    internal.append(("WARN", f"the 95% interval spans more than a factor of 10 "
                             f"({TAU_CI_LO:.0f} - {TAU_CI_HI:.0f} d) - a weak measurement"))
else:
    internal.append(("PASS", f"95% interval is a factor of "
                             f"{TAU_CI_HI / TAU_CI_LO:.1f} wide"))

if VIEWING_RISK == "HIGH":
    internal.append(("FAIL", "viewing angle screen flagged this as jet-dominated - "
                             "tau is not a clean disc timescale"))
elif VIEWING_RISK in ("NORMAL", "UNKNOWN"):
    internal.append(("WARN", f"viewing angle risk is {VIEWING_RISK} - state this "
                             f"caveat in the write-up"))
else:
    internal.append(("PASS", "viewing angle screen came back LOW risk"))

if "NOT GOOD" in DATA_VERDICT:
    internal.append(("FAIL", f"cell 6 verdict was '{DATA_VERDICT}'"))
elif "CAVEAT" in DATA_VERDICT:
    internal.append(("WARN", f"cell 6 verdict was '{DATA_VERDICT}'"))
else:
    internal.append(("PASS", "cell 6 said the data is good"))

for status, text in internal:
    print(f"  [{status}] {text}")

# --- compare against previously published work -------------------------
print("\n" + "=" * 70)
print("CONFIRM AGAINST PAST RESULTS")
print("=" * 70)
print("a number is only defensible once it's been put next to what other")
print("people got. search ADS / NED / the CRTS and ZTF variability papers for")
print(f"'{AGN_NAME}' and look for a published DRW tau, a structure-function")
print("timescale, or a reverberation-mapping lag for a comparable object.\n")

_lit = input("published / previous tau in days (Enter to skip): ").strip()
LIT_TAU = None
LIT_SOURCE = ""
if _lit:
    try:
        LIT_TAU = float(_lit)
        LIT_SOURCE = input("where is that from? (author, year, or a note): ").strip()
    except ValueError:
        print("   -> couldn't read that as a number, skipping the comparison")

if LIT_TAU:
    ratio = FIT["tau"] / LIT_TAU
    inside_ci = (TAU_CI_LO is not None and TAU_CI_LO <= LIT_TAU <= TAU_CI_HI)
    print(f"\n  our tau        : {FIT['tau']:.1f} days")
    print(f"  published tau  : {LIT_TAU:.1f} days"
          f"{f' ({LIT_SOURCE})' if LIT_SOURCE else ''}")
    print(f"  ratio          : {ratio:.2f}x")
    if inside_ci:
        LIT_VERDICT = ("CONSISTENT - the published value falls inside our 95% "
                       "confidence interval")
    elif 1 / 3 <= ratio <= 3:
        LIT_VERDICT = ("BROADLY CONSISTENT - within a factor of 3, which is "
                       "normal scatter for DRW timescales measured from "
                       "different baselines and bands")
    else:
        LIT_VERDICT = (f"INCONSISTENT - off by {ratio:.1f}x. go back through "
                       f"cells 5-8 before claiming anything: check the band, "
                       f"the cleaning cuts, and whether the published number "
                       f"uses the Kelly 2009 sigma convention instead of ours")
    print(f"\n  verdict: {LIT_VERDICT}")
else:
    LIT_VERDICT = ("no published comparison done yet - find one before this goes "
                   "in the report")
    print(f"\n  {LIT_VERDICT}")
print("=" * 70)

print("\n" + "-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  why compare at all: a number nobody has checked against anything is")
print("     not yet a result. if your tau lands near what others found for")
print("     this object or similar ones, that's evidence the pipeline works.")
print("     if it's wildly different, that's a signal to go back through the")
print("     earlier cells - not a discovery.")
print()
print("  where to look: NASA ADS (ui.adsabs.harvard.edu) is the search engine")
print("     for astronomy papers. search the object name plus terms like")
print("     'variability', 'damped random walk', 'structure function' or")
print("     'reverberation mapping'. MacLeod et al. (2010) and Burke et al.")
print("     (2021) are good starting points for typical tau values.")
print()
print("  fair comparisons: tau depends on the band and on the rest frame. a")
print("     published value from a different filter, or one not corrected for")
print("     redshift when yours isn't, won't match exactly. within a factor of")
print("     a few is normal agreement in this subfield.")
print()
print("  before declaring disagreement, check the sigma convention (see the")
print("     notes in cell 8) and check whether the published tau is in the")
print("     observed frame or the rest frame.")
print()
print("  reading the internal checks above: PASS means that particular concern")
print("     is handled. WARN means report it as a caveat. FAIL means the number")
print("     should not be presented as a disc timescale without heavy")
print("     qualification - go and address it.")
print()
print("  next: cell 10, measure and correct the fitting bias")
print("-" * 70)

## Cell 10 - Bias correction

Injects light curves with known timescales at this exact cadence and recovers them blind, to measure how far off the fit tends to be - then corrects for it. This is the slow cell.

In [ ]:
# ==============================================================
# CELL 10 - bias correction by injection and recovery
# ==============================================================
#
# A maximum-likelihood tau is biased low when tau isn't small compared to
# the baseline. Rather than trusting a rule of thumb, we measure the bias
# for THIS light curve: inject fake DRW curves with a known tau at the
# exact same epochs and error bars, recover them with the exact same
# fitting procedure, and see how far off the recovery lands.
#
# The recovery fits use a fixed, blind starting grid - never seeded near
# the injected tau. Seeding the optimizer at the true answer makes the
# whole test circular and it will happily "recover" anything.

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

if "FIT" not in globals():
    raise RuntimeError("run cell 8 first - FIT isn't defined yet")

print("this simulates and refits hundreds of light curves - it is the slow")
print("part of the notebook. 15 sims per point takes a few minutes, 30 is")
print("better but roughly doubles the wait.\n")

_answer = input("run the bias correction now? (y/n) [y]: ").strip().lower() or "y"

if not _answer.startswith("y"):
    BIAS_APPLIED = False
    BIAS_TABLE = None
    TAU_CORRECTED = None
    BIAS_PLOT_PATH = None
    print("\nskipped. the final report will carry the raw tau with a note that")
    print("no bias correction was applied.")
else:
    _n = input("simulations per injected tau [15]: ").strip()
    N_SIMS = int(_n) if _n.isdigit() and int(_n) > 0 else 15

    IR_TAU0_GRID = [50, 300, 1000, 3000]    # blind - independent of the injected value
    IR_SIGMA0_GRID = [0.05, 0.2]

    def simulate_drw(t, tau, sigma, mean, rng):
        """exact step-by-step DRW, works with irregular sampling."""
        y = np.empty(len(t))
        y[0] = rng.normal(mean, sigma)
        for i in range(1, len(t)):
            a = np.exp(-(t[i] - t[i - 1]) / tau)
            y[i] = rng.normal(mean + a * (y[i - 1] - mean), sigma * np.sqrt(1 - a ** 2))
        return y

    RATIOS = np.geomspace(0.01, 2.5, 10)    # injected tau / baseline
    rng = np.random.default_rng(0)
    rows = []

    print(f"\nrunning {len(RATIOS)} injected timescales x {N_SIMS} simulations "
          f"= {len(RATIOS) * N_SIMS} fits\n")

    for k, ratio in enumerate(RATIOS, start=1):
        tau_true = ratio * BASELINE
        recovered = []
        for _ in range(N_SIMS):
            y_sim = simulate_drw(T, tau_true, FIT["sigma"], FIT["mean"], rng)
            y_sim = y_sim + rng.normal(0.0, YERR)
            f = fit_drw(T, y_sim, YERR, tau0_grid=IR_TAU0_GRID, sigma0_grid=IR_SIGMA0_GRID)
            recovered.append(f["tau"])
            rows.append({"ratio_true": ratio, "tau_true": tau_true,
                         "tau_rec": f["tau"], "ratio_rec": f["tau"] / BASELINE})
        print(f"  [{k:>2}/{len(RATIOS)}] injected tau = {tau_true:>8.0f} d "
              f"(tau/T = {ratio:.3f})  ->  median recovered = "
              f"{np.median(recovered):>8.0f} d")

    INJECTION_RECOVERY = pd.DataFrame(rows)
    BIAS_TABLE = (INJECTION_RECOVERY.groupby("ratio_true")["ratio_rec"]
                  .agg(median="median",
                       p16=lambda x: np.percentile(x, 16),
                       p84=lambda x: np.percentile(x, 84))
                  .reset_index())
    BIAS_TABLE["recovery_fraction"] = BIAS_TABLE["median"] / BIAS_TABLE["ratio_true"]

    print("\n" + "=" * 70)
    print("BIAS TABLE - recovered tau/baseline vs injected tau/baseline")
    print("=" * 70)
    display(BIAS_TABLE.round(4))

    # invert the curve: what injected tau would typically be measured as ours?
    _bt = BIAS_TABLE.sort_values("median")
    if _bt["median"].min() <= TAU_OVER_BASELINE <= _bt["median"].max():
        corrected_ratio = float(np.interp(TAU_OVER_BASELINE,
                                          _bt["median"].to_numpy(),
                                          _bt["ratio_true"].to_numpy()))
        TAU_CORRECTED = corrected_ratio * BASELINE
        BIAS_APPLIED = True
    else:
        corrected_ratio = None
        TAU_CORRECTED = None
        BIAS_APPLIED = False

    # a point estimate alone is misleading once the curve saturates, so also
    # ask: which injected timescales could plausibly have produced what we
    # measured? that's every row whose 16-84 percentile band contains it.
    _consistent = BIAS_TABLE[(BIAS_TABLE["p16"] <= TAU_OVER_BASELINE)
                             & (BIAS_TABLE["p84"] >= TAU_OVER_BASELINE)]
    if not _consistent.empty:
        TAU_CORRECTED_LO = float(_consistent["ratio_true"].min()) * BASELINE
        TAU_CORRECTED_HI = float(_consistent["ratio_true"].max()) * BASELINE
        CORRECTION_UNBOUNDED = bool(
            _consistent["ratio_true"].max() >= BIAS_TABLE["ratio_true"].max() * 0.999
        )
    else:
        TAU_CORRECTED_LO = TAU_CORRECTED_HI = None
        CORRECTION_UNBOUNDED = False

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(BIAS_TABLE["ratio_true"], BIAS_TABLE["median"], "-o",
            color="steelblue", label="median recovered")
    ax.fill_between(BIAS_TABLE["ratio_true"], BIAS_TABLE["p16"], BIAS_TABLE["p84"],
                    color="steelblue", alpha=0.25, label="16th - 84th percentile")
    lim = float(BIAS_TABLE["ratio_true"].max())
    ax.plot([0, lim], [0, lim], "k--", alpha=0.6, label="perfect recovery (y = x)")
    ax.axhline(TAU_OVER_BASELINE, color="crimson", linestyle=":",
               label=f"our measured tau/T = {TAU_OVER_BASELINE:.3f}")
    if corrected_ratio is not None:
        ax.axvline(corrected_ratio, color="darkgreen", linestyle=":",
                   label=f"implied true tau/T = {corrected_ratio:.3f}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("injected tau / baseline")
    ax.set_ylabel("recovered tau / baseline")
    ax.set_title(f"{AGN_NAME} - injection & recovery at this exact cadence")
    ax.legend(loc="best", fontsize=9)
    ax.grid(alpha=0.3, which="both")
    plt.tight_layout()

    BIAS_PLOT_PATH = os.path.join(SAVE_DIR, f"{_safe}_bias_curve.png")
    plt.savefig(BIAS_PLOT_PATH, dpi=200)
    plt.show()

    print("=" * 70)
    print("BIAS CORRECTION")
    print("=" * 70)
    print(f"  measured tau              : {FIT['tau']:.1f} days "
          f"(tau/T = {TAU_OVER_BASELINE:.4f})")
    if BIAS_APPLIED:
        print(f"  bias-corrected tau        : {TAU_CORRECTED:.1f} days "
              f"(tau/T = {corrected_ratio:.4f})")
        print(f"  correction factor         : {TAU_CORRECTED / FIT['tau']:.2f}x")
        if TAU_CORRECTED_LO is not None:
            _hi = ("unbounded" if CORRECTION_UNBOUNDED
                   else f"{TAU_CORRECTED_HI:.0f} days")
            print(f"  plausible true tau range  : {TAU_CORRECTED_LO:.0f} days to {_hi}")
        print("\n  read this as: a source whose true timescale is the corrected")
        print("  value would typically be measured as our value, given this")
        print("  cadence, baseline and noise level.")
        if CORRECTION_UNBOUNDED:
            print("\n  CAUTION: timescales all the way to the top of the simulated")
            print("  range could have produced our measurement. past roughly")
            print("  tau/T = 0.1 the recovery curve flattens out, so the point")
            print("  estimate above is not well determined - quote the corrected")
            print("  value as a LOWER LIMIT, not a measurement.")
        elif TAU_CORRECTED / FIT["tau"] > 2:
            print("\n  CAUTION: this is a large correction. it means the baseline")
            print("  is short compared to the timescale, so treat the corrected")
            print("  number as indicative rather than precise.")
    else:
        print("  bias-corrected tau        : NOT APPLIED")
        print("\n  our measured tau/baseline falls outside the range the")
        print("  simulation covers, so inverting the curve would be")
        print("  extrapolation. report the raw number with this caveat stated.")
    print("=" * 70)

# notes print whether or not you ran the correction
print("\n" + "-" * 70)
print("NOTES FOR YOU")
print("-" * 70)
print("  the idea in one sentence: we can't ask the real AGN what its true")
print("     timescale is, but we can build fake light curves where we DO know")
print("     the answer, run them through the identical fitting code, and see")
print("     how far off it lands.")
print()
print("  why the fakes use your real epochs and your real error bars: bias")
print("     depends on the sampling. a light curve with seasonal gaps and this")
print("     particular noise level has its own bias curve, and a generic one")
print("     from a paper wouldn't apply to it.")
print()
print("  what 'blind' means and why it matters: every recovery fit starts from")
print("     the same fixed grid of guesses, never from the value we injected.")
print("     if you seed the optimiser at the true answer it will happily")
print("     return it and the test proves nothing. this is an easy mistake to")
print("     make and a hard one to spot in someone else's code.")
print()
print("  how to read the bias curve: the dashed diagonal is perfect recovery.")
print("     where the blue line falls below it, the fit is underestimating.")
print("     the curve flattens once the injected timescale gets comparable to")
print("     the baseline - past that point the data simply cannot tell long")
print("     timescales apart, which is why a measured value there maps to a")
print("     wide range of possible true values.")
print()
print("  this step is what separates a rigorous project from a typical one.")
print("     most undergraduate variability projects stop at cell 8 and report")
print("     the raw maximum-likelihood number as though it were unbiased.")
print()
print("  if it was too slow: re-run with fewer simulations for a rough answer")
print("     while testing, then do one careful run with 30 or more for the")
print("     final report.")
print()
print("  next: cell 11, compile everything for the write-up")
print("-" * 70)

## Cell 11 - Compiled result for the report

Every number and plot in one place, plus a write-up block you can paste straight into the project document. Saves everything to your download folder.

In [ ]:
# ==============================================================
# CELL 11 - the compiled result, ready to paste into the report
# ==============================================================
import os
from datetime import date

import numpy as np
import pandas as pd

try:
    from IPython.display import display, Image
except Exception:
    display = print
    Image = None

if "FIT" not in globals():
    raise RuntimeError("run cell 8 first - there's nothing to compile yet")

_bias_applied = globals().get("BIAS_APPLIED", False)
_tau_corrected = globals().get("TAU_CORRECTED", None)
_bias_table = globals().get("BIAS_TABLE", None)
_bias_plot = globals().get("BIAS_PLOT_PATH", None)
_lit_tau = globals().get("LIT_TAU", None)
_lit_source = globals().get("LIT_SOURCE", "")
_lit_verdict = globals().get("LIT_VERDICT", "no literature comparison done")

_unbounded = globals().get("CORRECTION_UNBOUNDED", False)
_corr_lo = globals().get("TAU_CORRECTED_LO", None)
_corr_hi = globals().get("TAU_CORRECTED_HI", None)

_ci_text = (f"{TAU_CI_LO:.1f} - {TAU_CI_HI:.1f} days"
            if TAU_CI_LO is not None else "unconstrained")
_excess = float(np.sqrt(max(NIGHTLY["mag"].std() ** 2
                            - np.mean(NIGHTLY["magerr"] ** 2), 0)))

# ---------------- the numbers, as one table ----------------
SUMMARY = pd.DataFrame([
    ("Target", AGN_NAME),
    ("Analysis date", OBS_DATE),
    ("RA (deg)", f"{RA_DEG}"),
    ("Dec (deg)", f"{DEC_DEG}"),
    ("Search radius (deg)", f"{RADIUS_DEG}  ({RADIUS_DEG * 3600:.2f} arcsec)"),
    ("Archive", "IRSA ZTF light curve API"),
    ("Band", BAND),
    ("ZTF object id", str(TARGET_OID)),
    ("Viewing-angle risk", VIEWING_RISK),
    ("Raw epochs returned", len(RAW)),
    ("Epochs passing quality cuts", len(CLEAN)),
    ("Nightly binned points fitted", len(NIGHTLY)),
    ("Baseline (days)", f"{BASELINE:.2f}"),
    ("Baseline (years)", f"{BASELINE / 365.25:.2f}"),
    ("Median error bar (mag)", f"{NIGHTLY['magerr'].median():.4f}"),
    ("Observed scatter (mag)", f"{NIGHTLY['mag'].std():.4f}"),
    ("Intrinsic variability (mag)", f"{_excess:.4f}"),
    ("Data quality verdict", DATA_VERDICT),
    ("Mean magnitude", f"{FIT['mean']:.4f}"),
    ("DRW sigma (mag)", f"{FIT['sigma']:.4f}"),
    ("DRW tau, measured (days)", f"{FIT['tau']:.2f}"),
    ("DRW tau, measured (years)", f"{FIT['tau'] / 365.25:.2f}"),
    ("95% CI on tau", _ci_text),
    ("tau / baseline", f"{TAU_OVER_BASELINE:.4f}"),
    ("Bias correction applied", "yes" if _bias_applied else "no"),
    ("DRW tau, bias-corrected (days)",
     f"{_tau_corrected:.2f}" if _bias_applied else "not applied"),
    ("Plausible true tau range (days)",
     (f"{_corr_lo:.0f} to " + ("unbounded" if _unbounded else f"{_corr_hi:.0f}"))
     if (_bias_applied and _corr_lo is not None) else "n/a"),
    ("Published tau compared against",
     f"{_lit_tau:.1f} days ({_lit_source})" if _lit_tau else "none yet"),
], columns=["Quantity", "Value"])

print("=" * 72)
print(f"FINAL COMPILED RESULT - {AGN_NAME}")
print("=" * 72)
display(SUMMARY)

# ---------------- the same thing as pasteable text ----------------
_headline_tau = _tau_corrected if _bias_applied else FIT["tau"]
_headline_label = "bias-corrected" if _bias_applied else "measured (no bias correction)"

if _bias_applied:
    _factor = _tau_corrected / FIT["tau"]
    _direction = "longer than" if _factor > 1 else "shorter than"
    _bias_text = (
        "Injection-recovery simulations were run at this exact cadence, sampling "
        "and noise level, injecting known timescales and recovering them with an "
        "identical, blind fitting procedure. Inverting the resulting bias curve "
        f"gives a bias-corrected timescale of {_tau_corrected:.1f} days "
        f"({_tau_corrected / 365.25:.2f} years), a factor of {_factor:.2f} "
        f"{_direction} the raw maximum-likelihood value."
    )
    if _corr_lo is not None:
        if _unbounded:
            _bias_text += (
                f" Injected timescales from {_corr_lo:.0f} days upwards are all "
                "consistent with what we measured, because the recovery curve "
                "flattens beyond tau/baseline of roughly 0.1. The corrected value "
                "should therefore be quoted as a lower limit rather than a "
                "measurement."
            )
        else:
            _bias_text += (
                f" Injected timescales between {_corr_lo:.0f} and {_corr_hi:.0f} "
                "days are consistent with our measurement at the 16th-84th "
                "percentile level."
            )
elif _bias_table is not None:
    _bias_text = (
        "No bias correction was applied - the measured tau/baseline fell outside "
        "the range covered by the injection-recovery simulation, so inverting the "
        "curve would have required extrapolation."
    )
else:
    _bias_text = "No bias correction was applied (cell 10 was skipped)."

if _bias_applied and _unbounded:
    _headline_label = "bias-corrected LOWER LIMIT"

report = f"""
## {AGN_NAME} - optical variability timescale from ZTF

**Analysis date:** {OBS_DATE}

### Data
Light curve retrieved from the IRSA ZTF light curve API by cone search at
RA = {RA_DEG} deg, Dec = {DEC_DEG} deg with a search radius of {RADIUS_DEG} deg
({RADIUS_DEG * 3600:.2f} arcsec), in the {BAND} band, saved as {FILE_FORMAT.upper()}.
The query returned {len(RAW)} epochs across {len(RAW.columns)} columns, of which
{len(CLEAN)} passed the quality cuts and {len(NIGHTLY)} remained after binning
same-night repeat exposures. The resulting light curve spans {BASELINE:.0f} days
({BASELINE / 365.25:.2f} years), with a median photometric uncertainty of
{NIGHTLY['magerr'].median():.4f} mag.

### Target suitability
The viewing-angle screen returned **{VIEWING_RISK} risk**.
{chr(10).join('- ' + r for r in VIEWING_REASONS) if VIEWING_REASONS else '- no classification or radio information was available'}

### Cleaning applied
Rows with missing mjd, mag or magerr were dropped; epochs carrying the ZTF
catflags bad-quality bit (32768) were rejected; points with magerr >= {MAX_MAGERR} mag
were removed; a {MAD_NSIGMA}-sigma clip against the median using the median absolute
deviation removed outliers; and same-night repeat exposures were combined into
a single inverse-variance weighted point per night. Data quality verdict:
{DATA_VERDICT}.

### Model
The light curve was modelled as a damped random walk (an Ornstein-Uhlenbeck
process), with covariance k(dt) = sigma^2 exp(-dt/tau) in the celerite /
MacLeod et al. (2010) convention. Parameters were estimated by maximum
likelihood using celerite2, started from {len(TAU0_GRID) * len(SIGMA0_GRID)} different
initial guesses to avoid local minima.

### Result
- DRW amplitude sigma = {FIT['sigma']:.4f} mag
- DRW damping timescale tau = {FIT['tau']:.1f} days ({FIT['tau'] / 365.25:.2f} years)
- 95% confidence interval (profile likelihood): {_ci_text}
- tau / baseline = {TAU_OVER_BASELINE:.4f}

### Bias correction
{_bias_text}

### Comparison with previous work
{_lit_verdict}

### Headline number
**tau = {_headline_tau:.0f} days ({_headline_tau / 365.25:.2f} years)** - {_headline_label}, 95% CI {_ci_text}.
""".strip()

print("\n" + "=" * 72)
print("PASTE THIS STRAIGHT INTO THE PROJECT REPORT")
print("=" * 72)
print(report)
print("=" * 72)

# ---------------- save everything next to the data ----------------
_stamp = date.today().isoformat()
summary_path = os.path.join(SAVE_DIR, f"{_safe}_summary.csv")
report_path = os.path.join(SAVE_DIR, f"{_safe}_report_{_stamp}.md")
nightly_path = os.path.join(SAVE_DIR, f"{_safe}_nightly_lightcurve.csv")

SUMMARY.to_csv(summary_path, index=False)
NIGHTLY.to_csv(nightly_path, index=False)
with open(report_path, "w") as f:
    f.write(report + "\n")

saved = [summary_path, nightly_path, report_path, LC_PLOT_PATH]
if _bias_table is not None:
    bias_path = os.path.join(SAVE_DIR, f"{_safe}_bias_table.csv")
    _bias_table.to_csv(bias_path, index=False)
    saved.append(bias_path)
if _bias_plot:
    saved.append(_bias_plot)

print("\nfiles written:")
for p in saved:
    print(f"  {p}")

# ---------------- show the figures again, together ----------------
print("\nfigures for the report:")
if Image is not None:
    display(Image(filename=LC_PLOT_PATH))
    if _bias_plot:
        display(Image(filename=_bias_plot))
else:
    print(f"  light curve : {LC_PLOT_PATH}")
    if _bias_plot:
        print(f"  bias curve  : {_bias_plot}")

if VIEWING_RISK == "HIGH":
    print("\nREMINDER: the viewing-angle screen flagged this target as")
    print("jet-dominated. present the tau as a jet-contaminated measurement,")
    print("not as an accretion disc timescale.")

print("\n" + "-" * 72)
print("NOTES FOR YOU")
print("-" * 72)
print("  what you now have: a summary table, a write-up block you can paste")
print("     straight into the report, the binned light curve as a CSV, and")
print("     both figures at print resolution. all of it is in your download")
print("     folder alongside the original data.")
print()
print("  the write-up block is a starting point, not a finished section.")
print("     rewrite it in your own words - you'll be asked to defend every")
print("     sentence, and defending someone else's phrasing is harder than")
print("     defending your own.")
print()
print("  quote the honest number. if a bias correction was applied, lead with")
print("     the corrected value and give the raw one too. if the correction")
print("     came out as a lower limit, say 'greater than' rather than quoting")
print("     a precise figure you can't support.")
print()
print("  always report alongside tau: the confidence interval, the baseline,")
print("     tau/baseline, and the viewing-angle verdict. a tau on its own is")
print("     not a result anyone can evaluate.")
print()
print("  to run another target: go back to cell 2 and start again. nothing in")
print("     this notebook is tied to a particular object, and the files are")
print("     named after the target so earlier runs won't be overwritten.")
print()
print("  if several of you are working on different AGN, this is the point to")
print("     compare: same pipeline, same cuts, same conventions, so the")
print("     numbers are directly comparable across the group.")
print("-" * 72)